# **The SparkSession Entry point**

In [1]:
from pyspark.sql import SparkSession

In [2]:
# 1. Initialize or retrieve the SparkSession (the entry point for Spark SQL and DataFrame work)
spark = (SparkSession.builder                    # Use the builder pattern to configure the session
         .appName("Analyzing the vocabulary of Pride and Prejudice.") # Name your Spark application (visible in Spark UI)
         .getOrCreate())                         # Get an existing session or create a new one if none exists

# 2. Set the logging verbosity level of the Spark Context to "ALL"
# This displays every log message (DEBUG, INFO, WARN, ERROR) for deep debugging and monitoring
spark.sparkContext.setLogLevel("WARN")

# 3. Reference the underlying SparkContext object
# Accessing this attribute returns the low-level SparkContext, which manages cluster resources and RDDs
spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/09 21:10:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=local[*] appName=Analyzing the vocabulary of Pride and Prejudice.>

- **🔍 Step-by-Step Breakdown**
1. **SparkSession.builder ... .getOrCreate():**
    - In PySpark, everything starts with a SparkSession. The builder pattern allows you to set configurations (like app name or master URL) cleanly. .getOrCreate() ensures that if a session is already running in your script, it reuses it rather than crashing or creating a duplicate.
2. **spark.sparkContext.setLogLevel("ALL"):**
    - The sparkContext is the heart of the older RDD-level API. Setting the log level to "ALL" forces Spark to print every single internal log event to your console. This is very useful when learning or debugging why a job behaves a certain way, though it can make your console quite noisy!
3. **spark.sparkContext:**
    - This final line simply evaluates and references the SparkContext object itself, showing its object representation (displaying details like the app ID, master node, and UI web URL).

![](../images/Log_level.png)
![](../images/Log_level1.png)

# **Ingest and explore**

In [3]:
book = spark.read.text("../data/gutenberg_books/1342-0.txt")

book

DataFrame[value: string]

![](../images/Book_schema.png)

In [4]:
book.printSchema()

print(book.dtypes)

root
 |-- value: string (nullable = true)

[('value', 'string')]


In [5]:
print(spark.__doc__)

The entry point to programming Spark with the Dataset and DataFrame API.

    A SparkSession can be used to create :class:`DataFrame`, register :class:`DataFrame` as
    tables, execute SQL over tables, cache tables, and read parquet files.
    To create a :class:`SparkSession`, use the following builder pattern:

    .. versionchanged:: 3.4.0
        Supports Spark Connect.

    .. autoattribute:: builder
       :annotation:

    Examples
    --------
    Create a Spark session.

    >>> spark = (
    ...     SparkSession.builder
    ...         .master("local")
    ...         .appName("Word Count")
    ...         .config("spark.some.config.option", "some-value")
    ...         .getOrCreate()
    ... )

    Create a Spark session with Spark Connect.

    >>> spark = (
    ...     SparkSession.builder
    ...         .remote("sc://localhost")
    ...         .appName("Word Count")
    ...         .config("spark.some.config.option", "some-value")
    ...         .getOrCreate()
    ..

In [6]:
spark?

Type:        SparkSession
String form: <pyspark.sql.session.SparkSession object at 0x77d5848ffda0>
File:        /usr/local/lib/python3.12/site-packages/pyspark/sql/session.py
Docstring:  
The entry point to programming Spark with the Dataset and DataFrame API.

A SparkSession can be used to create :class:`DataFrame`, register :class:`DataFrame` as
tables, execute SQL over tables, cache tables, and read parquet files.
To create a :class:`SparkSession`, use the following builder pattern:

.. versionchanged:: 3.4.0
    Supports Spark Connect.

.. autoattribute:: builder
   :annotation:

Examples
--------
Create a Spark session.

>>> spark = (
...     SparkSession.builder
...         .master("local")
...         .appName("Word Count")
...         .config("spark.some.config.option", "some-value")
...         .getOrCreate()
... )

Create a Spark session with Spark Connect.

>>> spark = (
...     SparkSession.builder
...         .remote("sc://localhost")
...         .appName("Word Count")
...

In [7]:
book.show()

+--------------------+
|               value|
+--------------------+
|The Project Guten...|
|                    |
|This eBook is for...|
|almost no restric...|
|re-use it under t...|
|with this eBook o...|
|                    |
|                    |
|Title: Pride and ...|
|                    |
| Author: Jane Austen|
|                    |
|Posting Date: Aug...|
|Release Date: Jun...|
|Last Updated: Mar...|
|                    |
|   Language: English|
|                    |
|Character set enc...|
|                    |
+--------------------+
only showing top 20 rows


In [8]:
book.show(10, truncate=50)

+--------------------------------------------------+
|                                             value|
+--------------------------------------------------+
|The Project Gutenberg EBook of Pride and Prejud...|
|                                                  |
|This eBook is for the use of anyone anywhere at...|
|almost no restrictions whatsoever.  You may cop...|
|re-use it under the terms of the Project Gutenb...|
|    with this eBook or online at www.gutenberg.org|
|                                                  |
|                                                  |
|                        Title: Pride and Prejudice|
|                                                  |
+--------------------------------------------------+
only showing top 10 rows


In [9]:
book.show(5, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------
 value | The Project Gutenberg EBook of Pride and Prejudice, by Jane Austen   
-RECORD 1---------------------------------------------------------------------
 value |                                                                      
-RECORD 2---------------------------------------------------------------------
 value | This eBook is for the use of anyone anywhere at no cost and with     
-RECORD 3---------------------------------------------------------------------
 value | almost no restrictions whatsoever.  You may copy it, give it away or 
-RECORD 4---------------------------------------------------------------------
 value | re-use it under the terms of the Project Gutenberg License included  
only showing top 5 rows


## **📌 Summary**

By default, PySpark uses **lazy evaluation**, meaning data frame operations are only planned, not executed, until an explicit action (like `.show()`) is triggered. Users coming from libraries like `pandas` or R often find this frustrating because referencing a variable prints its structural schema rather than the actual data. However, while PySpark allows you to enable **eager evaluation** (evaluating data frames immediately after every step) for learning or demonstrations, doing so sacrifices the deep query optimizations that give Spark its high performance.

---

## **🔍 Detailed Explanation**

### **1. The Shock for Pandas & R Users**
* In **pandas** or R data frames, evaluation is *eager*: every operation runs immediately, and calling the variable prints a preview of the data.
* In **PySpark**, calling a variable or running transformations (like filtering or adding columns) returns structural metadata instantly because **no data work is actually being performed yet**—it's all stored as a lazy execution plan.

### **2. Why PySpark Forces Explicit Actions (`show()`)**

* PySpark treats methods like `.show()` as **actions** because they require visible data output.
* Making actions explicit protects developers from accidentally triggering heavy, time-consuming distributed computations mid-pipeline.
* Keeping transformations lazy gives Spark's **Catalyst Optimizer** the full picture of your pipeline before execution, allowing it to reorder and optimize the code for maximum efficiency.

### **3. Turning on "Nonlazy" Spark (Eager Evaluation)**

* Since Spark 2.4.0, you can configure your `SparkSession` to evaluate data frames automatically after every instruction by setting a configuration flag:
```python
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.config(
        "spark.sql.repl.eagerEval.enabled", "True"
    )
    .getOrCreate()
)

```

* **When to use it:** This mode is helpful when you are learning PySpark or live-demonstrating code in a shell and want immediate visual feedback.

### **4. The Performance Warning**
* While eager evaluation mimics pandas, it **disables lazy evaluation**. 
* Because Spark loses its ability to view and optimize the entire pipeline as a single unit, you will be "leaving some extra horsepower on the table" in terms of performance and speed.

# **column transformations**

In [10]:
from pyspark.sql.functions import split

In [11]:
# 1. Transform the 'book' DataFrame to split text lines into arrays of words
# book.select(...) This is a transformation. It takes your source DataFrame (book) and applies a column-level transformation.
lines = book.select(
    # split() takes the string column 'book.value' and splits it into an array wherever there is a space (" ")
    # .alias("line") renames the resulting column to 'line'
    split(book.value, " ").alias("line")
)

# 2. Trigger an Action to execute the lazy plan and display the results
# .show(5) prints the first 5 rows of the DataFrame
# truncate=False ensures that long arrays or text strings are fully displayed without cutting them off with "..."
lines.show(5, truncate=False)

+----------------------------------------------------------------------------------+
|line                                                                              |
+----------------------------------------------------------------------------------+
|[The, Project, Gutenberg, EBook, of, Pride, and, Prejudice,, by, Jane, Austen]    |
|[]                                                                                |
|[This, eBook, is, for, the, use, of, anyone, anywhere, at, no, cost, and, with]   |
|[almost, no, restrictions, whatsoever., , You, may, copy, it,, give, it, away, or]|
|[re-use, it, under, the, terms, of, the, Project, Gutenberg, License, included]   |
+----------------------------------------------------------------------------------+
only showing top 5 rows


In [12]:
book.select(book.value).show(5)

+--------------------+
|               value|
+--------------------+
|The Project Guten...|
|                    |
|This eBook is for...|
|almost no restric...|
|re-use it under t...|
+--------------------+
only showing top 5 rows


In [13]:
from pyspark.sql.functions import col

In [14]:
print(book.select(book.value))
print(book.select(book["value"]))
print(book.select(col("value")))
print(book.select("value"))

DataFrame[value: string]
DataFrame[value: string]
DataFrame[value: string]
DataFrame[value: string]


In [15]:
lines = book.select(split(col("value"), " "))

lines.printSchema()

root
 |-- split(value,  , -1): array (nullable = true)
 |    |-- element: string (containsNull = false)



In [16]:
lines.show(5, truncate=False)

+----------------------------------------------------------------------------------+
|split(value,  , -1)                                                               |
+----------------------------------------------------------------------------------+
|[The, Project, Gutenberg, EBook, of, Pride, and, Prejudice,, by, Jane, Austen]    |
|[]                                                                                |
|[This, eBook, is, for, the, use, of, anyone, anywhere, at, no, cost, and, with]   |
|[almost, no, restrictions, whatsoever., , You, may, copy, it,, give, it, away, or]|
|[re-use, it, under, the, terms, of, the, Project, Gutenberg, License, included]   |
+----------------------------------------------------------------------------------+
only showing top 5 rows


### **🏛️ Summary: PySpark's Architecture & JVM Heritage**

* **Direct JVM Delegation:** Core PySpark functions (such as `split()`) do not rewrite functionality natively in Python. Instead, they use a translation layer to invoke underlying Java Virtual Machine (JVM) functions directly via `sc._jvm` objects.
* **Performance & Portability Benefits:**
* **Speed:** It enhances performance by avoiding the overhead of constantly translating Python code into JVM equivalents.
* **Portability:** It makes porting PySpark to other platforms easier because developers do not have to re-implement entire libraries from scratch.


* **Key Trade-offs:** Relying on the JVM heritage means PySpark's built-in functions utilize **JVM-based regular expressions** rather than native Python regular expressions.

In [17]:
lines = book.select(split(col("value"), " "))
lines1 = book.select(split(col("value"), " ").alias("line"))

lines.printSchema()
lines1.printSchema()

root
 |-- split(value,  , -1): array (nullable = true)
 |    |-- element: string (containsNull = false)

root
 |-- line: array (nullable = true)
 |    |-- element: string (containsNull = false)



- When you’re using a method where you’re specifying which columns you want to appear, like the **select(**) method, use **alias()**.
- If you just want to rename a column without changing the rest of the data frame, use **.withColumnRenamed**. Note that, should the column not exist, PySpark will treat this method as a no-op and not perform anything.

![](../images/explode_method.png)

In [18]:
from pyspark.sql.functions import explode

In [19]:
words = lines1.select(explode(col("line")).alias("words"))

words.show()

+----------+
|     words|
+----------+
|       The|
|   Project|
| Gutenberg|
|     EBook|
|        of|
|     Pride|
|       and|
|Prejudice,|
|        by|
|      Jane|
|    Austen|
|          |
|      This|
|     eBook|
|        is|
|       for|
|       the|
|       use|
|        of|
|    anyone|
+----------+
only showing top 20 rows


In [20]:
from pyspark.sql.functions import lower

In [21]:
lower_word = words.select(lower(col("words")).alias("lower word"))

lower_word.show()

+----------+
|lower word|
+----------+
|       the|
|   project|
| gutenberg|
|     ebook|
|        of|
|     pride|
|       and|
|prejudice,|
|        by|
|      jane|
|    austen|
|          |
|      this|
|     ebook|
|        is|
|       for|
|       the|
|       use|
|        of|
|    anyone|
+----------+
only showing top 20 rows


In [22]:
from pyspark.sql.functions import regexp_extract

In [23]:
clean_word = lower_word.select(regexp_extract(col("lower word"), "[a-z]+", 0).alias("word"))

clean_word.show()

+---------+
|     word|
+---------+
|      the|
|  project|
|gutenberg|
|    ebook|
|       of|
|    pride|
|      and|
|prejudice|
|       by|
|     jane|
|   austen|
|         |
|     this|
|    ebook|
|       is|
|      for|
|      the|
|      use|
|       of|
|   anyone|
+---------+
only showing top 20 rows


In [24]:
from pyspark.sql.functions import filter

In [25]:
words_nonull = clean_word.filter(col("word") != "")

words_nonull.show()

+---------+
|     word|
+---------+
|      the|
|  project|
|gutenberg|
|    ebook|
|       of|
|    pride|
|      and|
|prejudice|
|       by|
|     jane|
|   austen|
|     this|
|    ebook|
|       is|
|      for|
|      the|
|      use|
|       of|
|   anyone|
| anywhere|
+---------+
only showing top 20 rows


In [26]:
clean_word.where(col("word") != "").show()

+---------+
|     word|
+---------+
|      the|
|  project|
|gutenberg|
|    ebook|
|       of|
|    pride|
|      and|
|prejudice|
|       by|
|     jane|
|   austen|
|     this|
|    ebook|
|       is|
|      for|
|      the|
|      use|
|       of|
|   anyone|
| anywhere|
+---------+
only showing top 20 rows


# **Additional Exercises**